<a href="https://colab.research.google.com/github/Ghhaidaa/Data-Engineering-for-AI-System-DAICO/blob/main/notebooks/01_02_data_ingestion_delta_lake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SDAIA** **Books** **Platform**

A data engineering platform for validating, streaming, and storing book registration data submitted by publishers across Saudi Arabia.

# Data Ingestion and Delta Lake

This notebook presents the implementation of the **Data Ingestion** and **Delta Lakehouse** components for the SDAIA Books Platform.

The implementation includes:

- Data validation using **Pydantic**.
- Real-time data streaming with **Apache Kafka** (Producer & Consumer).
- Delta Lake **Bronze**, **Silver**, and **Gold** layers.
- **MERGE (Upsert)** using `book_id` as the business key.
- **Schema Enforcement** demonstration to validate schema consistency.

This notebook corresponds to **Parts 1 & 2** of the project deliverables.

In [ ]:
!apt-get update -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
!apt-get install -y openjdk-17-jdk-headless

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-17-jdk-headless is already the newest version (17.0.19+10-1~22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 118 not upgraded.


In [ ]:
!pip install kafka-python pydantic faker

In [ ]:
import pandas as pd
import random
from faker import Faker

fake = Faker()

# Make results reproducible
Faker.seed(42)
random.seed(42)

In [ ]:
publisher_names = [
    "Jarir Publishing",
    "Obeikan Publishing",
    "King Fahad National Library",
    "Dar Al-Minhaj",
    "Saudi Research Publishing"
]

publisher_cities = [
    "Riyadh",
    "Jeddah",
    "Dammam",
    "Madinah",
    "Abha"
]

categories = [
    "Artificial Intelligence",
    "Data Science",
    "Cybersecurity",
    "Education",
    "Government",
    "Culture",
    "History"
]

languages = ["Arabic", "English"]

records = []

for book_id in range(1, 101):

    isbn = f"978603{random.randint(1000000,9999999)}"

    records.append({
        "book_id": book_id,
        "isbn": isbn,
        "title": fake.sentence(nb_words=4).replace(".", ""),
        "author": fake.name(),
        "publisher_name": random.choice(publisher_names),
        "publisher_city": random.choice(publisher_cities),
        "category": random.choice(categories),
        "language": random.choice(languages),
        "publication_year": random.randint(2015, 2026),
        "submission_date": fake.date_this_year().strftime("%Y-%m-%d")
    })

df = pd.DataFrame(records)

df.head()

,book_id,isbn,title,author,publisher_name,publisher_city,category,language,publication_year,submission_date
0,1,9786032867825,Agent every,Noah Rhodes,Jarir Publishing,Dammam,Data Science,Arabic,2017,2026-06-30
1,2,9786032719583,Opportunity all,David Guzman,Saudi Research Publishing,Riyadh,Government,English,2015,2026-04-24
2,3,9786031499914,Information last everything thank serve,Jay Ramirez,Jarir Publishing,Jeddah,Data Science,Arabic,2023,2026-02-13
3,4,9786034335942,Bill here grow gas,Andrew Stevens,Saudi Research Publishing,Madinah,Data Science,English,2024,2026-01-26
4,5,9786035667265,Bad fall pick those,Corey Jones,Jarir Publishing,Jeddah,Culture,English,2020,2026-05-14


In [ ]:
# Missing ISBN
df.loc[0, "isbn"] = None

# Empty title
df.loc[1, "title"] = ""

# Unsupported language
df.loc[2, "language"] = "French"

# Future publication year
df.loc[3, "publication_year"] = 2050

# Invalid date format
df.loc[4, "submission_date"] = "2026/15/40"

df.head()

,book_id,isbn,title,author,publisher_name,publisher_city,category,language,publication_year,submission_date
0,1,None,Agent every,Noah Rhodes,Jarir Publishing,Dammam,Data Science,Arabic,2017,2026-06-30
1,2,9786032719583,,David Guzman,Saudi Research Publishing,Riyadh,Government,English,2015,2026-04-24
2,3,9786031499914,Information last everything thank serve,Jay Ramirez,Jarir Publishing,Jeddah,Data Science,French,2023,2026-02-13
3,4,9786034335942,Bill here grow gas,Andrew Stevens,Saudi Research Publishing,Madinah,Data Science,English,2050,2026-01-26
4,5,9786035667265,Bad fall pick those,Corey Jones,Jarir Publishing,Jeddah,Culture,English,2020,2026/15/40


In [ ]:
df.to_csv("books_data.csv", index=False)

print("Dataset created successfully!")
print(f"Total records: {len(df)}")

Dataset created successfully!
Total records: 100


In [ ]:
from pydantic import BaseModel, Field, ValidationError, field_validator
from datetime import datetime

In [ ]:
class BookRecord(BaseModel):
    book_id: int = Field(gt=0)
    isbn: str
    title: str
    author: str
    publisher_name: str
    publisher_city: str
    category: str
    language: str
    publication_year: int
    submission_date: str

    @field_validator("isbn")
    @classmethod
    def validate_isbn(cls, value):
        if not value:
            raise ValueError("ISBN cannot be empty.")
        return value

    @field_validator("title")
    @classmethod
    def validate_title(cls, value):
        if not value.strip():
            raise ValueError("Title cannot be empty.")
        return value

    @field_validator("language")
    @classmethod
    def validate_language(cls, value):
        if value not in ["Arabic", "English"]:
            raise ValueError("Language must be Arabic or English.")
        return value

    @field_validator("publication_year")
    @classmethod
    def validate_year(cls, value):
        current_year = datetime.now().year
        if value > current_year:
            raise ValueError("Publication year cannot be in the future.")
        return value

    @field_validator("submission_date")
    @classmethod
    def validate_date(cls, value):
        datetime.strptime(value, "%Y-%m-%d")
        return value

In [ ]:
valid_records = []
invalid_records = []

for _, row in df.iterrows():
    try:
        record = BookRecord(**row.to_dict())
        valid_records.append(record.model_dump())
    except ValidationError as e:
        invalid_records.append({
            **row.to_dict(),
            "error": str(e)
        })

valid_df = pd.DataFrame(valid_records)
invalid_df = pd.DataFrame(invalid_records)

print(f"✅ Valid Records: {len(valid_df)}")
print(f"❌ Invalid Records: {len(invalid_df)}")

✅ Valid Records: 95
❌ Invalid Records: 5


In [ ]:
valid_df.to_csv("valid_books.csv", index=False)
invalid_df.to_csv("rejected_books.csv", index=False)

print("Files saved successfully!")

print(f"Valid records: {len(valid_df)}")
print(f"Invalid records: {len(invalid_df)}")

Files saved successfully!
Valid records: 95
Invalid records: 5


In [ ]:
!wget -q https://archive.apache.org/dist/kafka/3.7.0/kafka_2.13-3.7.0.tgz
!tar -xzf kafka_2.13-3.7.0.tgz

^C

gzip: stdin: unexpected end of file
tar: Unexpected EOF in archive
tar: Unexpected EOF in archive
tar: Error is not recoverable: exiting now


In [ ]:
%cd kafka_2.13-3.7.0

/content/kafka_2.13-3.7.0


In [ ]:
import os

cluster_id = os.popen("bin/kafka-storage.sh random-uuid").read().strip()

print(cluster_id)

In [ ]:
!bin/kafka-storage.sh format -t $cluster_id -c config/kraft/server.properties

Error: Could not find or load main class kafka.tools.StorageTool
Caused by: java.lang.ClassNotFoundException: kafka.tools.StorageTool


In [ ]:
!rm -rf kafka_2.13-3.7.0
!rm -f kafka_2.13-3.7.0.tgz

In [ ]:
!wget https://archive.apache.org/dist/kafka/3.7.0/kafka_2.13-3.7.0.tgz

--2026-07-22 15:40:35--  https://archive.apache.org/dist/kafka/3.7.0/kafka_2.13-3.7.0.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 119028138 (114M) [application/x-gzip]
Saving to: ‘kafka_2.13-3.7.0.tgz’

kafka_2.13-3.7.0.tg 100%[===================>] 113.51M   123KB/s    in 14m 49s 

2026-07-22 15:55:25 (131 KB/s) - ‘kafka_2.13-3.7.0.tgz’ saved [119028138/119028138]



In [ ]:
!tar -xzf kafka_2.13-3.7.0.tgz

In [ ]:
%cd kafka_2.13-3.7.0

/content/kafka_2.13-3.7.0/kafka_2.13-3.7.0


In [ ]:
import os

cluster_id = os.popen("bin/kafka-storage.sh random-uuid").read().strip()
print(cluster_id)

umQ5Ne1vQNqFsLcglYsEcA


In [ ]:
!bin/kafka-storage.sh format -t $cluster_id -c config/kraft/server.properties

metaPropertiesEnsemble=MetaPropertiesEnsemble(metadataLogDir=Optional.empty, dirs={/tmp/kraft-combined-logs: EMPTY})
Formatting /tmp/kraft-combined-logs with metadata.version 3.7-IV4.


In [ ]:
import subprocess
import time

broker = subprocess.Popen(
    ["bin/kafka-server-start.sh", "config/kraft/server.properties"]
)

time.sleep(10)

print("Kafka Broker started!")

In [ ]:
import subprocess
import time

broker = subprocess.Popen(
    ["bin/kafka-server-start.sh", "config/kraft/server.properties"]
)

time.sleep(10)

print("Kafka Broker started!")

Kafka Broker started!


In [ ]:
!bin/kafka-topics.sh \
  --create \
  --topic books \
  --bootstrap-server localhost:9092

Created topic books.


In [ ]:
from kafka import KafkaProducer
import pandas as pd
import json


# Step 1: Create a Kafka Producer
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda x: json.dumps(x).encode("utf-8")
)

# Step 2: Load the validated dataset
valid_books = pd.read_csv("/content/valid_books.csv")


# Step 3: Send each record to the Kafka topic
for _, record in valid_books.iterrows():
    producer.send("books", value=record.to_dict())


# Step 4: Ensure all messages are delivered
producer.flush()

print(f"Successfully sent {len(valid_books)} records to the 'books' topic.")

Successfully sent 95 records to the 'books' topic.


In [ ]:
from kafka import KafkaConsumer
import json


# Step 1: Create a Kafka Consumer
consumer = KafkaConsumer(
    "books",
    bootstrap_servers="localhost:9092",
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    value_deserializer=lambda x: json.loads(x.decode("utf-8"))
)

print("Reading records from Kafka...\n")

# Step 2: Read records from Kafka
count = 0

for message in consumer:
    print(message.value)
    count += 1

    # Stop after reading 95 records
    if count == 95:
        break

consumer.close()

print(f"\nSuccessfully consumed {count} records.")

/tmp/ipykernel_2932/2806510815.py:6: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Reading records from Kafka...

{'book_id': 6, 'isbn': 9786035661907, 'title': 'World talk term', 'author': 'Jesse Flowers', 'publisher_name': 'Obeikan Publishing', 'publisher_city': 'Jeddah', 'category': 'History', 'language': 'English', 'publication_year': 2016, 'submission_date': '2026-02-02'}
{'book_id': 7, 'isbn': 9786032556017, 'title': 'Decide environment view possible', 'author': 'Carolyn Daniel', 'publisher_name': 'Dar Al-Minhaj', 'publisher_city': 'Riyadh', 'category': 'Cybersecurity', 'language': 'English', 'publication_year': 2024, 'submission_date': '2026-02-03'}
{'book_id': 8, 'isbn': 9786035437923, 'title': 'Establish understand read detail', 'author': 'Randy Brown', 'publisher_name': 'Jarir Publishing', 'publisher_city': 'Madinah', 'category': 'Government', 'language': 'Arabic', 'publication_year': 2021, 'submission_date': '2026-06-16'}
{'book_id': 9, 'isbn': 9786032322047, 'title': 'Husband at tree note', 'author': 'Christy Porter', 'publisher_name': 'Saudi Research Pub

In [ ]:
!pip install -q pyspark==3.5.0 delta-spark==3.2.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 15.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.


In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("SDAIA Books Pipeline")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark Session is ready!")

Spark Session is ready!


In [ ]:
from pyspark.sql.functions import current_timestamp


# Step 1: Load the validated dataset
bronze_df = spark.read.csv(
    "/content/valid_books.csv",
    header=True,
    inferSchema=True
)


# Step 2: Add ingestion timestamp
bronze_df = bronze_df.withColumn(
    "ingestion_time",
    current_timestamp()
)


# Step 3: Save as Delta Bronze table
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/delta/bronze")

print("Bronze layer created successfully!")

Bronze layer created successfully!


In [ ]:
bronze = spark.read.format("delta").load("/content/delta/bronze")

print(f"Number of records: {bronze.count()}")

bronze.show(5, truncate=False)

Number of records: 95
+-------+-------------+--------------------------------+--------------+-------------------------+--------------+-------------+--------+----------------+---------------+--------------------------+
|book_id|isbn         |title                           |author        |publisher_name           |publisher_city|category     |language|publication_year|submission_date|ingestion_time            |
+-------+-------------+--------------------------------+--------------+-------------------------+--------------+-------------+--------+----------------+---------------+--------------------------+
|6      |9786035661907|World talk term                 |Jesse Flowers |Obeikan Publishing       |Jeddah        |History      |English |2016            |2026-02-02     |2026-07-22 16:13:55.682499|
|7      |9786032556017|Decide environment view possible|Carolyn Daniel|Dar Al-Minhaj            |Riyadh        |Cybersecurity|English |2024            |2026-02-03     |2026-07-22 16:13:55.682499

In [ ]:
from pyspark.sql.functions import col


# Step 1: Read Bronze layer
silver_df = spark.read.format("delta").load("/content/delta/bronze")


# Step 2: Clean the data
silver_df = (
    silver_df
    .dropDuplicates()
    .withColumn("publication_year", col("publication_year").cast("int"))
    .withColumn("submission_date", col("submission_date").cast("date"))
)


# Step 3: Save as Delta Silver table
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/delta/silver")

print("Silver layer created successfully!")

Silver layer created successfully!


In [ ]:
silver = spark.read.format("delta").load("/content/delta/silver")

print(f"Number of records: {silver.count()}")

silver.show(5, truncate=False)

Number of records: 95
+-------+-------------+---------------------------------+---------------+---------------------------+--------------+-----------------------+--------+----------------+---------------+--------------------------+
|book_id|isbn         |title                            |author         |publisher_name             |publisher_city|category               |language|publication_year|submission_date|ingestion_time            |
+-------+-------------+---------------------------------+---------------+---------------------------+--------------+-----------------------+--------+----------------+---------------+--------------------------+
|46     |9786035449368|Attorney professional form finish|Michelle Evans |Dar Al-Minhaj              |Dammam        |Education              |English |2017            |2026-03-04     |2026-07-22 16:13:55.682499|
|90     |9786037705062|Play man before girl             |Michael Hodge  |Obeikan Publishing         |Jeddah        |Culture               

In [ ]:
from pyspark.sql.functions import count


# Step 1: Read Silver layer
silver = spark.read.format("delta").load("/content/delta/silver")


# Step 2: Create Gold layer
# Count books by category and language
gold_df = (
    silver
    .groupBy("category", "language")
    .agg(count("*").alias("number_of_books"))
)


# Step 3: Save as Delta Gold table
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/delta/gold")

print("Gold layer created successfully!")

Gold layer created successfully!


In [ ]:
gold = spark.read.format("delta").load("/content/delta/gold")

gold.orderBy("category", "language").show(truncate=False)

+-----------------------+--------+---------------+
|category               |language|number_of_books|
+-----------------------+--------+---------------+
|Artificial Intelligence|Arabic  |8              |
|Artificial Intelligence|English |5              |
|Culture                |Arabic  |9              |
|Culture                |English |5              |
|Cybersecurity          |Arabic  |3              |
|Cybersecurity          |English |7              |
|Data Science           |Arabic  |5              |
|Data Science           |English |10             |
|Education              |Arabic  |8              |
|Education              |English |7              |
|Government             |Arabic  |8              |
|Government             |English |5              |
|History                |Arabic  |6              |
|History                |English |9              |
+-----------------------+--------+---------------+



In [42]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, to_date

# Read the schema from Silver
silver = spark.read.format("delta").load("/content/delta/silver")

updates = [
    (
        46,
        "9786035449368",
        "Updated Book Title",
        "Michelle Evans",
        "Dar Al-Minhaj",
        "Dammam",
        "Education",
        "English",
        2025,
        "2026-07-22"
    )
]

updates_df = spark.createDataFrame(
    updates,
    schema="""
    book_id INT,
    isbn STRING,
    title STRING,
    author STRING,
    publisher_name STRING,
    publisher_city STRING,
    category STRING,
    language STRING,
    publication_year INT,
    submission_date STRING
    """
)

updates_df = (
    updates_df
    .withColumn("submission_date", to_date("submission_date"))
    .withColumn("ingestion_time", current_timestamp())
)

updates_df.show()

+-------+-------------+------------------+--------------+--------------+--------------+---------+--------+----------------+---------------+--------------------+
|book_id|         isbn|             title|        author|publisher_name|publisher_city| category|language|publication_year|submission_date|      ingestion_time|
+-------+-------------+------------------+--------------+--------------+--------------+---------+--------+----------------+---------------+--------------------+
|     46|9786035449368|Updated Book Title|Michelle Evans| Dar Al-Minhaj|        Dammam|Education| English|            2025|     2026-07-22|2026-07-22 17:16:...|
+-------+-------------+------------------+--------------+--------------+--------------+---------+--------+----------------+---------------+--------------------+



MERGE

The following example demonstrates a Delta Lake MERGE (Upsert) operation using `book_id` as the business key.

In [43]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forPath(spark, "/content/delta/silver")

(
    silver_table.alias("target")
    .merge(
        updates_df.alias("source"),
        "target.book_id = source.book_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Merge completed successfully!")

Merge completed successfully!


In [44]:
silver = spark.read.format("delta").load("/content/delta/silver")

silver.filter("book_id = 46").show(truncate=False)

+-------+-------------+------------------+--------------+--------------+--------------+---------+--------+----------------+---------------+--------------------------+
|book_id|isbn         |title             |author        |publisher_name|publisher_city|category |language|publication_year|submission_date|ingestion_time            |
+-------+-------------+------------------+--------------+--------------+--------------+---------+--------+----------------+---------------+--------------------------+
|46     |9786035449368|Updated Book Title|Michelle Evans|Dar Al-Minhaj |Dammam        |Education|English |2025            |2026-07-22     |2026-07-22 17:18:04.802447|
+-------+-------------+------------------+--------------+--------------+--------------+---------+--------+----------------+---------------+--------------------------+



## Schema Enforcement
The following example demonstrates Delta Lake schema enforcement by attempting to write data that does not match the existing table schema.

In [45]:
from pyspark.sql import Row

# Create invalid data with an extra column
invalid_data = [
    Row(
        book_id=999,
        isbn="9786030000000",
        title="Invalid Book",
        author="Test Author",
        publisher_name="Test Publisher",
        publisher_city="Riyadh",
        category="Education",
        language="English",
        publication_year=2025,
        submission_date="2026-07-22",
        extra_column="This column should not exist"
    )
]

invalid_df = spark.createDataFrame(invalid_data)

try:
    invalid_df.write \
        .format("delta") \
        .mode("append") \
        .save("/content/delta/silver")

    print("Schema accepted.")

except Exception as e:
    print("Schema Enforcement is working!")
    print(e)

Schema Enforcement is working!
[DELTA_FAILED_TO_MERGE_FIELDS] Failed to merge fields 'book_id' and 'book_id'
